# Ретро-сравнение прогнозов кассовых показателей

Самодостаточный ноутбук для ежедневного ретро-прогона за период, заданный параметрами `RETRO_DATE_FROM` и `RETRO_DATE_TO`.

Для каждой даты скоринга модель:

- использует данные только до `score_date − 2`;
- строит прогноз на 30 календарных дней;
- для метрик использует только первый день горизонта;
- сравнивает новый прогноз `atdtmco_ns` со старым `forecast_model` на общей отрицательной шкале.

Последняя ячейка с записью в Oracle выключена по умолчанию.

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prefect.blocks.system import Secret
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX
from toolbox import oracle

warnings.filterwarnings("ignore")

## 1. Параметры ретро-прогона

История ограничена 12 месяцами. Для каждой даты скоринга последняя доступная дата — `T−2`; чтобы получить 30 прогнозных дней начиная с `T`, модель строит 31 шаг и отбрасывает технический день `T−1`.

In [ ]:
SOURCE_TABLE = "AIDA2.AIDA_TRS_DTM_CASHOP@aida"
OLD_FORECAST_TABLE = "AIDA2.AIDA_TRS_DTM_FORECAST_HISTORY@aida"
ORACLE_TARGET_TABLE = "EMA_CASHDESK_PREDS_2M"

RETRO_DATE_FROM = pd.Timestamp("2026-04-15")
RETRO_DATE_TO = pd.Timestamp("2026-06-15")
HISTORY_MONTHS = 12
ACTIVE_LOOKBACK_MONTHS = 1
DATA_LAG_DAYS = 2
FORECAST_DAYS = 30
MODEL_STEPS = FORECAST_DAYS + DATA_LAG_DAYS - 1

INITIAL_TRAIN_DAYS = 60
BACKTEST_STEP_DAYS = 7
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_QUANTILE = 0.15
MIN_BACKTEST_ERROR_BLOCKS = 5
MIN_SARIMA_DAYS = 90
SARIMA_NO_ERROR_HISTORY_ADJUSTMENT = 0.15
SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)
CASH_NEED_CLIP_UPPER = 0.0

EXCLUDE_DATE_RANGES = [("2025-09-01", "2025-11-01")]
CASHDESK_FILTER = None
RANDOM_SEED = 42

score_dates = pd.date_range(RETRO_DATE_FROM, RETRO_DATE_TO, freq="D")
MIN_REPORT_DATE = score_dates.min() - pd.Timedelta(days=DATA_LAG_DAYS)
DATA_DATE_FROM = MIN_REPORT_DATE - pd.DateOffset(months=HISTORY_MONTHS)
FACT_DATE_TO_EXCLUSIVE = score_dates.max() + pd.Timedelta(days=FORECAST_DAYS)
OLD_DATE_TO_EXCLUSIVE = RETRO_DATE_TO + pd.Timedelta(days=1)

LOAD_RAW_FROM_CACHE = False
RAW_CACHE_PATH = Path("data/raw/cashdesk_retro_2m_raw.parquet")
OLD_CACHE_PATH = Path("data/raw/cashdesk_old_forecast_2m.parquet")
OUTPUT_DIR = Path("forecast_results/retro_comparison")
PLOT_DIR = OUTPUT_DIR / "plots"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
USE_CHECKPOINTS = True

print(f"Дат скоринга: {len(score_dates)}")
print(f"История и факты: {DATA_DATE_FROM.date()} — {(FACT_DATE_TO_EXCLUSIVE - pd.Timedelta(days=1)).date()}")
print(f"Старый прогноз: {RETRO_DATE_FROM.date()} — {RETRO_DATE_TO.date()}")

## 2. Загрузка исходных данных и старого прогноза

Из основной таблицы загружается единый диапазон, достаточный и для 12-месячного обучения первой даты, и для фактов всех 30-дневных горизонтов. Старый прогноз нужен только для дат скоринга.

In [ ]:
USERNAME_CDW = "sb_analytics"
engine_cdw = None


async def create_cdw_engine():
    password_cdw = (await Secret.load("pass-sb-analytics")).get()
    return oracle.create_engine_cdw(USERNAME_CDW, password_cdw)


if LOAD_RAW_FROM_CACHE:
    raw_df = pd.read_parquet(RAW_CACHE_PATH)
    old_history_df = pd.read_parquet(OLD_CACHE_PATH)
    print("Данные прочитаны из локального parquet-кэша")
else:
    engine_cdw = await create_cdw_engine()

    source_query = f"""
    select
        atdtmco_cashdesk_name,
        atdtmco_cashdesk_name_trn,
        atdtmco_calday,
        atdtmco_saldo_turn,
        atdtmco_ns
    from {SOURCE_TABLE}
    where atdtmco_calday >= date '{DATA_DATE_FROM.date().isoformat()}'
      and atdtmco_calday < date '{FACT_DATE_TO_EXCLUSIVE.date().isoformat()}'
    """

    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        source_query += (
            f"\n  and not (atdtmco_calday >= date '{exclude_start}' "
            f"and atdtmco_calday < date '{exclude_end}')"
        )

    if CASHDESK_FILTER:
        escaped_names = ", ".join("'" + name.replace("'", "''") + "'" for name in CASHDESK_FILTER)
        source_query += f"\n  and atdtmco_cashdesk_name in ({escaped_names})"

    old_query = f"""
    select
        cashdesk_name,
        calday,
        flow_minimum,
        forecast_model,
        forecast_time
    from {OLD_FORECAST_TABLE}
    where calday >= date '{RETRO_DATE_FROM.date().isoformat()}'
      and calday < date '{OLD_DATE_TO_EXCLUSIVE.date().isoformat()}'
    """

    with engine_cdw.connect() as conn:
        raw_df = pd.read_sql(text(source_query), conn)
        old_history_df = pd.read_sql(text(old_query), conn)

    RAW_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_CACHE_PATH, index=False)
    old_history_df.to_parquet(OLD_CACHE_PATH, index=False)
    print(f"Кэш сохранён: {RAW_CACHE_PATH}, {OLD_CACHE_PATH}")

raw_df.columns = raw_df.columns.str.lower()
old_history_df.columns = old_history_df.columns.str.lower()
raw_df["atdtmco_calday"] = pd.to_datetime(raw_df["atdtmco_calday"]).dt.normalize()
old_history_df["calday"] = pd.to_datetime(old_history_df["calday"]).dt.normalize()
old_history_df["forecast_time"] = pd.to_datetime(old_history_df["forecast_time"], errors="coerce")

print(f"Загружено: {len(raw_df):,} исходных строк и {len(old_history_df):,} старых прогнозов")

## 3. Дневная агрегация и контроль качества склейки

`flow_minimum` проверяется против сырого дневного минимума NS. Для модели используется тот же минимум, ограниченный сверху нулём. Дубликаты старой истории на кассу и дату сворачиваются до последней записи по `forecast_time`.

In [ ]:
daily_df = (
    raw_df
    .sort_values(["atdtmco_cashdesk_name", "atdtmco_calday"])
    .groupby(
        ["atdtmco_cashdesk_name", "atdtmco_calday"],
        as_index=False,
        dropna=False,
    )
    .agg(
        atdtmco_cashdesk_name_trn=("atdtmco_cashdesk_name_trn", "last"),
        atdtmco_saldo_turn_fact=("atdtmco_saldo_turn", "sum"),
        atdtmco_ns_daily_min_raw=("atdtmco_ns", "min"),
    )
    .rename(columns={"atdtmco_calday": "calday"})
    .sort_values(["atdtmco_cashdesk_name", "calday"])
    .reset_index(drop=True)
)
daily_df["atdtmco_ns_fact"] = daily_df["atdtmco_ns_daily_min_raw"].clip(upper=0.0)

old_history_dedup_df = (
    old_history_df
    .sort_values("forecast_time")
    .drop_duplicates(["cashdesk_name", "calday"], keep="last")
    .reset_index(drop=True)
)

translation_name_counts = daily_df.groupby("atdtmco_cashdesk_name_trn")[
    "atdtmco_cashdesk_name"
].nunique()
ambiguous_translated_names = set(translation_name_counts[translation_name_counts > 1].index)

flow_check_df = old_history_dedup_df[
    ["cashdesk_name", "calday", "flow_minimum"]
].merge(
    daily_df[
        ["atdtmco_cashdesk_name_trn", "calday", "atdtmco_ns_daily_min_raw"]
    ],
    left_on=["cashdesk_name", "calday"],
    right_on=["atdtmco_cashdesk_name_trn", "calday"],
    how="inner",
)
flow_minimum_match_rate = np.isclose(
    flow_check_df["atdtmco_ns_daily_min_raw"],
    flow_check_df["flow_minimum"],
    atol=0.01,
    rtol=0.0,
).mean()
print(f"Совпадение дневного min(NS) с flow_minimum: {flow_minimum_match_rate:.1%}")

## 4. Модельное ядро: SARIMA, rolling backtest и bootstrap

Для потока используется центральный SARIMA-прогноз. Для потребности к центральному прогнозу добавляются целые исторические 31-дневные блоки ошибок rolling backtest; из 1000 сценариев берётся нижний уровень 0.15. При короткой истории или ошибке SARIMA применяется fallback по дню недели.

In [ ]:
@dataclass(frozen=True)
class SarimaConfig:
    order: Tuple[int, int, int]
    seasonal_order: Tuple[int, int, int, int]
    maxiter: int = 200


SARIMA_CONFIG = SarimaConfig(
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
)


def make_regular_daily_series(
    cashdesk_df: pd.DataFrame,
    value_col: str,
    report_date: pd.Timestamp,
) -> pd.Series:
    observed = (
        cashdesk_df
        .set_index("calday")[value_col]
        .sort_index()
        .astype(float)
    )
    full_index = pd.date_range(observed.index.min(), report_date, freq="D")
    series = observed.reindex(full_index).fillna(0.0)
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        excluded_mask = (
            (series.index >= pd.Timestamp(exclude_start))
            & (series.index < pd.Timestamp(exclude_end))
        )
        series.loc[excluded_mask] = np.nan
    series.index.name = "calday"
    return series


def fit_sarima(y: pd.Series, config: SarimaConfig):
    model = SARIMAX(
        y,
        order=config.order,
        seasonal_order=config.seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    return model.fit(disp=False, maxiter=config.maxiter)


def make_future_index(y: pd.Series, steps: int) -> pd.DatetimeIndex:
    return pd.date_range(y.index.max() + pd.Timedelta(days=1), periods=steps, freq="D")


def sarima_point_forecast(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
    output_col: str,
) -> pd.DataFrame:
    fitted = fit_sarima(y, config)
    forecast_mean = fitted.get_forecast(steps=steps).predicted_mean
    output = pd.DataFrame({output_col: forecast_mean.to_numpy()}, index=forecast_mean.index)
    output.index.name = "forecast_date"
    return output.reset_index()


def weekday_point_forecast(y: pd.Series, steps: int, output_col: str) -> pd.DataFrame:
    future_index = make_future_index(y, steps)
    y_clean = y.dropna()
    global_value = float(y_clean.median())
    weekday_values = y_clean.groupby(y_clean.index.dayofweek).median()
    values = [weekday_values.get(day.dayofweek, global_value) for day in future_index]
    return pd.DataFrame({"forecast_date": future_index, output_col: values})


def weekday_quantile_forecast(
    y: pd.Series,
    steps: int,
    quantile: float,
    output_col: str,
) -> pd.DataFrame:
    future_index = make_future_index(y, steps)
    y_clean = y.dropna()
    global_quantile = float(y_clean.quantile(quantile))
    weekday_quantiles = y_clean.groupby(y_clean.index.dayofweek).quantile(quantile)
    values = np.array(
        [weekday_quantiles.get(day.dayofweek, global_quantile) for day in future_index],
        dtype=float,
    )
    if CASH_NEED_CLIP_UPPER is not None:
        values = np.minimum(values, CASH_NEED_CLIP_UPPER)
    return pd.DataFrame({"forecast_date": future_index, output_col: values})


def collect_backtest_error_blocks(
    y: pd.Series,
    steps: int,
    initial_train_days: int,
    step_days: int,
    config: SarimaConfig,
) -> np.ndarray:
    error_blocks = []
    max_train_end = len(y) - steps

    for train_end in range(initial_train_days, max_train_end + 1, step_days):
        train = y.iloc[:train_end]
        test = y.iloc[train_end:train_end + steps]
        if train.notna().sum() < initial_train_days:
            continue
        try:
            fitted = fit_sarima(train, config)
            forecast = fitted.get_forecast(steps=steps).predicted_mean.to_numpy()
            error = test.to_numpy() - forecast
            if np.isfinite(error).all():
                error_blocks.append(error)
        except Exception:
            continue

    if not error_blocks:
        return np.empty((0, steps))
    return np.vstack(error_blocks)


def forecast_with_error_bootstrap(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
    output_col: str,
    random_generator: np.random.Generator,
) -> pd.DataFrame:
    error_blocks = collect_backtest_error_blocks(
        y=y,
        steps=steps,
        initial_train_days=INITIAL_TRAIN_DAYS,
        step_days=BACKTEST_STEP_DAYS,
        config=config,
    )
    fitted = fit_sarima(y, config)
    forecast_mean = fitted.get_forecast(steps=steps).predicted_mean
    mean_values = np.minimum(
        forecast_mean.to_numpy(copy=True), CASH_NEED_CLIP_UPPER
    )

    if len(error_blocks) < MIN_BACKTEST_ERROR_BLOCKS:
        # NS хранится на отрицательной шкале: множитель > 1 сдвигает прогноз
        # в более консервативную сторону, дальше от нуля.
        forecast_values = mean_values * (1 + SARIMA_NO_ERROR_HISTORY_ADJUSTMENT)
    else:
        sampled_indexes = random_generator.integers(
            low=0,
            high=len(error_blocks),
            size=BOOTSTRAP_ITERATIONS,
        )
        scenarios = np.minimum(
            mean_values.reshape(1, -1) + error_blocks[sampled_indexes],
            CASH_NEED_CLIP_UPPER,
        )
        forecast_values = np.quantile(scenarios, BOOTSTRAP_QUANTILE, axis=0)

    forecast_values = np.minimum(forecast_values, CASH_NEED_CLIP_UPPER)
    return pd.DataFrame(
        {"forecast_date": forecast_mean.index, output_col: forecast_values}
    )


def forecast_cashdesk(
    cashdesk_df: pd.DataFrame,
    report_date: pd.Timestamp,
    steps: int,
    random_generator: np.random.Generator,
) -> pd.DataFrame:
    saldo_series = make_regular_daily_series(
        cashdesk_df, "atdtmco_saldo_turn_fact", report_date
    )
    ns_series = make_regular_daily_series(cashdesk_df, "atdtmco_ns_fact", report_date)

    try:
        if saldo_series.notna().sum() < MIN_SARIMA_DAYS:
            raise ValueError("short_history")
        saldo_forecast = sarima_point_forecast(
            saldo_series, steps, SARIMA_CONFIG, "atdtmco_saldo_turn_pred"
        )
    except Exception:
        saldo_forecast = weekday_point_forecast(
            saldo_series, steps, "atdtmco_saldo_turn_pred"
        )

    try:
        if ns_series.notna().sum() < MIN_SARIMA_DAYS:
            raise ValueError("short_history")
        ns_forecast = forecast_with_error_bootstrap(
            y=ns_series,
            steps=steps,
            config=SARIMA_CONFIG,
            output_col="atdtmco_ns_pred",
            random_generator=random_generator,
        )
    except Exception:
        ns_forecast = weekday_quantile_forecast(
            ns_series, steps, BOOTSTRAP_QUANTILE, "atdtmco_ns_pred"
        )

    return saldo_forecast.merge(ns_forecast, on="forecast_date", how="inner")

## 5. Ежедневный ретро-прогон

Для каждой даты из `score_dates` используется отдельное скользящее окно глубиной `HISTORY_MONTHS`. Активной считается касса, у которой была хотя бы одна исходная строка за месяц до `report_date`. Прогнозы строятся на весь заданный горизонт; факты присоединяются только после обучения.

Каждая готовая дата сохраняется в `CHECKPOINT_DIR`. При повторном запуске она загружается без пересчёта. Если параметры модели изменились, установите `USE_CHECKPOINTS = False` для полного пересчёта.

In [ ]:
def run_retro_scoring_day(
    score_date: pd.Timestamp,
    all_daily_df: pd.DataFrame,
    random_generator: np.random.Generator,
) -> pd.DataFrame:
    score_date = pd.Timestamp(score_date).normalize()
    report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    history_date_from = report_date - pd.DateOffset(months=HISTORY_MONTHS)
    active_date_from = report_date - pd.DateOffset(months=ACTIVE_LOOKBACK_MONTHS)
    forecast_date_to_exclusive = score_date + pd.Timedelta(days=FORECAST_DAYS)

    available_df = all_daily_df[
        (all_daily_df["calday"] >= history_date_from)
        & (all_daily_df["calday"] <= report_date)
    ].copy()
    active_cashdesks = (
        available_df.loc[
            available_df["calday"] >= active_date_from,
            "atdtmco_cashdesk_name",
        ]
        .dropna()
        .drop_duplicates()
    )
    train_df = available_df[
        available_df["atdtmco_cashdesk_name"].isin(active_cashdesks)
    ].copy()

    result_parts = []

    for cashdesk_name, cashdesk_df in train_df.groupby(
        "atdtmco_cashdesk_name", sort=True
    ):
        cashdesk_df = cashdesk_df.sort_values("calday")
        translated_names = cashdesk_df["atdtmco_cashdesk_name_trn"].dropna()
        translated_name = translated_names.iloc[-1] if len(translated_names) else pd.NA

        try:
            forecast_df = forecast_cashdesk(
                cashdesk_df=cashdesk_df,
                report_date=report_date,
                steps=MODEL_STEPS,
                random_generator=random_generator,
            )
            forecast_df = forecast_df[
                (forecast_df["forecast_date"] >= score_date)
                & (forecast_df["forecast_date"] < forecast_date_to_exclusive)
            ].copy()
            forecast_df.insert(0, "score_date", score_date)
            forecast_df.insert(1, "report_date", report_date)
            forecast_df.insert(2, "atdtmco_cashdesk_name", cashdesk_name)
            forecast_df.insert(3, "atdtmco_cashdesk_name_trn", translated_name)
            result_parts.append(forecast_df)
        except Exception as exc:
            print(f"Пропущена касса {cashdesk_name}: {exc}")

    forecast_result_df = pd.concat(result_parts, ignore_index=True)
    actual_df = (
        all_daily_df[
            (all_daily_df["calday"] >= score_date)
            & (all_daily_df["calday"] < forecast_date_to_exclusive)
        ][
            [
                "atdtmco_cashdesk_name",
                "atdtmco_cashdesk_name_trn",
                "calday",
                "atdtmco_saldo_turn_fact",
                "atdtmco_ns_daily_min_raw",
                "atdtmco_ns_fact",
            ]
        ]
        .rename(columns={"calday": "forecast_date"})
    )
    forecast_result_df = forecast_result_df.merge(
        actual_df,
        on=["atdtmco_cashdesk_name", "forecast_date"],
        how="left",
        suffixes=("", "_fact"),
    )
    forecast_result_df["atdtmco_cashdesk_name_trn"] = forecast_result_df[
        "atdtmco_cashdesk_name_trn_fact"
    ].combine_first(forecast_result_df["atdtmco_cashdesk_name_trn"])
    forecast_result_df = forecast_result_df.drop(
        columns="atdtmco_cashdesk_name_trn_fact"
    )
    fact_columns = [
        "atdtmco_saldo_turn_fact",
        "atdtmco_ns_daily_min_raw",
        "atdtmco_ns_fact",
    ]
    forecast_result_df[fact_columns] = forecast_result_df[fact_columns].fillna(0.0)

    return forecast_result_df


CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
retro_result_parts = []

for score_index, score_date in enumerate(score_dates, start=1):
    checkpoint_path = CHECKPOINT_DIR / f"retro_{score_date:%Y-%m-%d}.parquet"
    if USE_CHECKPOINTS and checkpoint_path.exists():
        print(f"[{score_index:02d}/{len(score_dates)}] загружено: {score_date.date()}")
        day_result_df = pd.read_parquet(checkpoint_path)
    else:
        print(f"[{score_index:02d}/{len(score_dates)}] расчёт: {score_date.date()}")
        day_rng = np.random.default_rng(RANDOM_SEED + score_date.toordinal())
        day_result_df = run_retro_scoring_day(score_date, daily_df, day_rng)
        temporary_path = checkpoint_path.with_suffix(".tmp.parquet")
        day_result_df.to_parquet(temporary_path, index=False)
        temporary_path.replace(checkpoint_path)
    retro_result_parts.append(day_result_df)

retro_result_df = pd.concat(retro_result_parts, ignore_index=True)
print(f"Итог: {len(retro_result_df):,} строк, {retro_result_df['atdtmco_cashdesk_name'].nunique():,} касс")

## 6. First-day выборка и сравнение со старым алгоритмом

В метрики попадает только `forecast_date = score_date`. Старый положительный `forecast_model` переводится на отрицательную шкалу как `-forecast_model`. Неоднозначные английские названия исключаются из сравнения.

In [ ]:
first_day_df = retro_result_df[
    retro_result_df["forecast_date"] == retro_result_df["score_date"]
].copy()
first_day_for_compare_df = first_day_df[
    ~first_day_df["atdtmco_cashdesk_name_trn"].isin(ambiguous_translated_names)
].copy()
old_new_compare_df = first_day_for_compare_df.merge(
    old_history_dedup_df[["cashdesk_name", "calday", "forecast_model"]],
    left_on=["atdtmco_cashdesk_name_trn", "forecast_date"],
    right_on=["cashdesk_name", "calday"],
    how="inner",
).dropna(subset=["forecast_model"])
if old_new_compare_df.empty:
    raise RuntimeError(
        "Не найдено ни одной однозначно сопоставленной строки со старым forecast_model"
    )

old_new_compare_df["atdtmco_ns_pred_old"] = -pd.to_numeric(
    old_new_compare_df["forecast_model"], errors="coerce"
)
old_new_compare_df["error_atdtmco_ns_new"] = (
    old_new_compare_df["atdtmco_ns_fact"] - old_new_compare_df["atdtmco_ns_pred"]
)
old_new_compare_df["error_atdtmco_ns_old"] = (
    old_new_compare_df["atdtmco_ns_fact"] - old_new_compare_df["atdtmco_ns_pred_old"]
)
old_new_compare_df["ns_breach_new"] = (
    old_new_compare_df["atdtmco_ns_fact"] < old_new_compare_df["atdtmco_ns_pred"]
)
old_new_compare_df["ns_breach_old"] = (
    old_new_compare_df["atdtmco_ns_fact"] < old_new_compare_df["atdtmco_ns_pred_old"]
)
old_new_compare_df["is_working_day"] = ~(
    old_new_compare_df["atdtmco_saldo_turn_fact"].eq(0)
    & old_new_compare_df["atdtmco_ns_fact"].eq(0)
)
first_day_df["is_working_day"] = ~(
    first_day_df["atdtmco_saldo_turn_fact"].eq(0)
    & first_day_df["atdtmco_ns_fact"].eq(0)
)

## 7. Устойчивые метрики

`MAE95` для NS исключает 5% наиболее отрицательных фактических значений потребности. Для потока исключаются верхние 5% по `abs(факт)`. Отбор зависит только от факта, поэтому новая и старая модели сравниваются на одинаковых строках.

In [ ]:
def median_smoothness_ratio(
    df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    ratios = []
    for _, group in df.groupby("atdtmco_cashdesk_name"):
        ordered = group.sort_values("forecast_date")
        consecutive_mask = ordered["forecast_date"].diff().dt.days.eq(1)
        fact_change = ordered[fact_col].diff().abs()[consecutive_mask]
        pred_change = ordered[pred_col].diff().abs()[consecutive_mask]
        fact_total_variation = fact_change.sum()
        if fact_total_variation > 0:
            ratios.append(pred_change.sum() / fact_total_variation)
    return float(np.median(ratios)) if ratios else np.nan


def build_fact_trim_mask(
    clean_df: pd.DataFrame,
    target: str,
    fact_col: str,
) -> Tuple[pd.Series, float]:
    trim_count = int(np.floor(len(clean_df) * 0.05))
    trim_mask = pd.Series(True, index=clean_df.index)

    if target == "atdtmco_ns":
        ordered_indexes = clean_df[fact_col].sort_values(
            ascending=True, kind="mergesort"
        ).index
        fact_trim_threshold = clean_df[fact_col].quantile(0.05)
    else:
        ordered_indexes = clean_df[fact_col].abs().sort_values(
            ascending=False, kind="mergesort"
        ).index
        fact_trim_threshold = clean_df[fact_col].abs().quantile(0.95)

    if trim_count:
        trim_mask.loc[ordered_indexes[:trim_count]] = False
    return trim_mask, fact_trim_threshold


def build_metric_row(
    df: pd.DataFrame,
    target: str,
    model: str,
    segment: str,
    fact_col: str,
    pred_col: str,
) -> dict:
    clean_df = df.dropna(subset=[fact_col, pred_col]).copy()
    absolute_error = (clean_df[fact_col] - clean_df[pred_col]).abs()
    trim_mask, fact_trim_threshold = build_fact_trim_mask(clean_df, target, fact_col)

    if target == "atdtmco_ns":
        breach_rate = (clean_df[fact_col] < clean_df[pred_col]).mean()
        wape = np.nan
    else:
        breach_rate = np.nan
        denominator = clean_df[fact_col].abs().sum()
        wape = absolute_error.sum() / denominator if denominator > 0 else np.nan

    trimmed_absolute_error = absolute_error[trim_mask]
    return {
        "target": target,
        "model": model,
        "segment": segment,
        "rows": len(clean_df),
        "cashdesks": clean_df["atdtmco_cashdesk_name"].nunique(),
        "score_dates": clean_df["score_date"].nunique(),
        "fact_mean": clean_df[fact_col].mean(),
        "forecast_mean": clean_df[pred_col].mean(),
        "mae": absolute_error.mean(),
        "mae95": trimmed_absolute_error.mean(),
        "mae95_rows": int(trim_mask.sum()),
        "fact_trim_threshold": fact_trim_threshold,
        "median_absolute_error": absolute_error.median(),
        "p90_absolute_error": absolute_error.quantile(0.90),
        "bias_fact_minus_forecast": (clean_df[fact_col] - clean_df[pred_col]).mean(),
        "wape": wape,
        "breach_rate": breach_rate,
        "median_smoothness_ratio": median_smoothness_ratio(clean_df, fact_col, pred_col),
    }


metric_rows = []
for segment, working_only in (("all_days", False), ("working_days", True)):
    ns_segment_df = (
        old_new_compare_df[old_new_compare_df["is_working_day"]].copy()
        if working_only
        else old_new_compare_df.copy()
    )
    metric_rows.append(
        build_metric_row(
            ns_segment_df,
            target="atdtmco_ns",
            model="new_sarima_bootstrap",
            segment=segment,
            fact_col="atdtmco_ns_fact",
            pred_col="atdtmco_ns_pred",
        )
    )
    metric_rows.append(
        build_metric_row(
            ns_segment_df,
            target="atdtmco_ns",
            model="old_forecast_model",
            segment=segment,
            fact_col="atdtmco_ns_fact",
            pred_col="atdtmco_ns_pred_old",
        )
    )
    saldo_segment_df = (
        first_day_df[first_day_df["is_working_day"]].copy()
        if working_only
        else first_day_df.copy()
    )
    metric_rows.append(
        build_metric_row(
            saldo_segment_df,
            target="atdtmco_saldo_turn",
            model="new_sarima",
            segment=segment,
            fact_col="atdtmco_saldo_turn_fact",
            pred_col="atdtmco_saldo_turn_pred",
        )
    )

overall_metrics_df = pd.DataFrame(metric_rows).assign(
    scope="overall",
    atdtmco_cashdesk_name="ALL",
)

cashdesk_metric_rows = []
for cashdesk_name, cashdesk_df in old_new_compare_df.groupby("atdtmco_cashdesk_name"):
    for segment, working_only in (("all_days", False), ("working_days", True)):
        segment_df = (
            cashdesk_df[cashdesk_df["is_working_day"]]
            if working_only
            else cashdesk_df
        )
        if segment_df.empty:
            continue
        for model, pred_col in (
            ("new_sarima_bootstrap", "atdtmco_ns_pred"),
            ("old_forecast_model", "atdtmco_ns_pred_old"),
        ):
            row = build_metric_row(
                segment_df,
                target="atdtmco_ns",
                model=model,
                segment=segment,
                fact_col="atdtmco_ns_fact",
                pred_col=pred_col,
            )
            row.update(scope="cashdesk", atdtmco_cashdesk_name=cashdesk_name)
            cashdesk_metric_rows.append(row)

for cashdesk_name, cashdesk_df in first_day_df.groupby("atdtmco_cashdesk_name"):
    for segment, working_only in (("all_days", False), ("working_days", True)):
        segment_df = (
            cashdesk_df[cashdesk_df["is_working_day"]]
            if working_only
            else cashdesk_df
        )
        if segment_df.empty:
            continue
        row = build_metric_row(
            segment_df,
            target="atdtmco_saldo_turn",
            model="new_sarima",
            segment=segment,
            fact_col="atdtmco_saldo_turn_fact",
            pred_col="atdtmco_saldo_turn_pred",
        )
        row.update(scope="cashdesk", atdtmco_cashdesk_name=cashdesk_name)
        cashdesk_metric_rows.append(row)

cashdesk_metrics_df = pd.DataFrame(cashdesk_metric_rows)
metrics_summary_df = pd.concat(
    [overall_metrics_df, cashdesk_metrics_df], ignore_index=True
)
worst_cashdesks_df = (
    cashdesk_metrics_df[
        (cashdesk_metrics_df["target"] == "atdtmco_ns")
        & (cashdesk_metrics_df["model"] == "new_sarima_bootstrap")
        & (cashdesk_metrics_df["segment"] == "all_days")
    ]
    .sort_values(["breach_rate", "mae95"], ascending=False)
    .head(20)
)

display(overall_metrics_df)
print("Кассы с наибольшей долей невыдач и MAE95:")
display(worst_cashdesks_df)

## 8. Наглядные графики и локальные результаты

Линии строятся по медиане касс за день, чтобы отдельные крупные кассы не скрывали общую динамику. Распределения дневных изменений показывают, насколько прогнозы сглажены относительно факта.

In [ ]:
def collect_consecutive_absolute_changes(
    df: pd.DataFrame,
    value_col: str,
) -> pd.Series:
    change_parts = []
    for _, group in df.groupby("atdtmco_cashdesk_name"):
        ordered = group.sort_values("forecast_date")
        consecutive_mask = ordered["forecast_date"].diff().dt.days.eq(1)
        changes = ordered[value_col].diff().abs()[consecutive_mask].dropna()
        if not changes.empty:
            change_parts.append(changes)
    if not change_parts:
        return pd.Series(dtype=float)
    return pd.concat(change_parts, ignore_index=True)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

ns_daily_plot_df = (
    old_new_compare_df
    .groupby("forecast_date", as_index=False)
    .agg(
        fact=("atdtmco_ns_fact", "median"),
        new=("atdtmco_ns_pred", "median"),
        old=("atdtmco_ns_pred_old", "median"),
        breach_new=("ns_breach_new", "mean"),
        breach_old=("ns_breach_old", "mean"),
    )
)
ns_error_new = old_new_compare_df["error_atdtmco_ns_new"].abs()
ns_error_old = old_new_compare_df["error_atdtmco_ns_old"].abs()
ns_error_plot_limit = pd.concat([ns_error_new, ns_error_old]).quantile(0.99)
ns_fact_changes = collect_consecutive_absolute_changes(old_new_compare_df, "atdtmco_ns_fact")
ns_new_changes = collect_consecutive_absolute_changes(old_new_compare_df, "atdtmco_ns_pred")
ns_old_changes = collect_consecutive_absolute_changes(old_new_compare_df, "atdtmco_ns_pred_old")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes[0, 0].plot(ns_daily_plot_df["forecast_date"], ns_daily_plot_df["fact"], label="Факт NS", linewidth=2)
axes[0, 0].plot(ns_daily_plot_df["forecast_date"], ns_daily_plot_df["new"], label="Новый", linewidth=2)
axes[0, 0].plot(ns_daily_plot_df["forecast_date"], ns_daily_plot_df["old"], label="Старый", linewidth=2)
axes[0, 0].set_title("Потребность кассы: медиана по кассам")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(ns_daily_plot_df["forecast_date"], 100 * ns_daily_plot_df["breach_new"], label="Новый")
axes[0, 1].plot(ns_daily_plot_df["forecast_date"], 100 * ns_daily_plot_df["breach_old"], label="Старый")
axes[0, 1].axhline(15, color="black", linestyle="--", linewidth=1, label="15%")
axes[0, 1].set_title("Доля невыдач по датам, %")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(ns_error_new.clip(upper=ns_error_plot_limit), bins=40, alpha=0.6, label="Новый")
axes[1, 0].hist(ns_error_old.clip(upper=ns_error_plot_limit), bins=40, alpha=0.6, label="Старый")
axes[1, 0].set_title("Абсолютная ошибка NS (для графика ограничена P99)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].boxplot(
    [ns_fact_changes, ns_new_changes, ns_old_changes],
    labels=["Факт", "Новый", "Старый"],
    showfliers=False,
)
axes[1, 1].set_title("Абсолютные дневные изменения NS")
axes[1, 1].grid(True, axis="y", alpha=0.3)

for axis in axes.flat:
    axis.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(PLOT_DIR / "ns_old_vs_new.png", dpi=160, bbox_inches="tight")
plt.show()

saldo_daily_plot_df = (
    first_day_df
    .groupby("forecast_date", as_index=False)
    .agg(
        fact=("atdtmco_saldo_turn_fact", "median"),
        forecast=("atdtmco_saldo_turn_pred", "median"),
    )
)
saldo_absolute_error = (
    first_day_df["atdtmco_saldo_turn_fact"] - first_day_df["atdtmco_saldo_turn_pred"]
).abs()
saldo_error_plot_limit = saldo_absolute_error.quantile(0.99)
saldo_fact_changes = collect_consecutive_absolute_changes(first_day_df, "atdtmco_saldo_turn_fact")
saldo_pred_changes = collect_consecutive_absolute_changes(first_day_df, "atdtmco_saldo_turn_pred")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(saldo_daily_plot_df["forecast_date"], saldo_daily_plot_df["fact"], label="Факт", linewidth=2)
axes[0].plot(saldo_daily_plot_df["forecast_date"], saldo_daily_plot_df["forecast"], label="Прогноз", linewidth=2)
axes[0].set_title("Дневной поток: медиана по кассам")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(saldo_absolute_error.clip(upper=saldo_error_plot_limit), bins=40)
axes[1].set_title("Абсолютная ошибка потока (ограничена P99)")
axes[1].grid(True, alpha=0.3)

axes[2].boxplot(
    [saldo_fact_changes, saldo_pred_changes],
    labels=["Факт", "Прогноз"],
    showfliers=False,
)
axes[2].set_title("Абсолютные дневные изменения потока")
axes[2].grid(True, axis="y", alpha=0.3)

for axis in axes:
    axis.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(PLOT_DIR / "saldo_turn_quality.png", dpi=160, bbox_inches="tight")
plt.show()

retro_result_df.to_csv(OUTPUT_DIR / "retro_forecasts_all_horizons.csv", index=False)
old_new_compare_df.to_csv(OUTPUT_DIR / "retro_first_day_comparison.csv", index=False)
metrics_summary_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)

print(f"Сохранены 3 итоговых файла в {OUTPUT_DIR.resolve()}")

## 9. Слайд: как устроен прогноз

### Что прогнозируем

- **`atdtmco_saldo_turn` — поток за день.** Для каждой кассы дневные значения суммируются; SARIMA учитывает динамику ряда и недельную сезонность.
- **`atdtmco_ns` — потребность кассы.** Используется минимальное значение за день, ограниченное сверху нулём.

### Схема

```text
12 месяцев истории, доступной на T−2
                  │
       дневные ряды по каждой кассе
                  │
        ┌─────────┴─────────┐
        │                   │
Поток за день          Потребность кассы
 saldo_turn                  NS
        │                   │
SARIMA с недельной      SARIMA: центральный
сезонностью 7 дней      прогноз на 30 дней
        │                   │
центральный прогноз     rolling backtest:
на 30 дней              факт − прогноз
                            │
                       1000 сценариев =
                       прогноз + блок ошибок
                            │
                       страховой прогноз,
                       симулирующий около
                       15% невыдач
```

### Ретро-проверка

Для каждой даты заданного периода модель строит полный прогнозный горизонт, не видя данные после `T−2`. Качество сравнивается по первому прогнозному дню: MAE, MAE без верхних 5% фактической потребности, bias, доля невыдач и сглаженность.

In [ ]:
# РУЧНОЙ ШАГ: переключить на True только после проверки результатов.
WRITE_TO_ORACLE = False

if WRITE_TO_ORACLE:
    oracle_export_df = retro_result_df[
        [
            "score_date",
            "report_date",
            "atdtmco_cashdesk_name",
            "forecast_date",
            "atdtmco_saldo_turn_pred",
            "atdtmco_ns_pred",
            "atdtmco_saldo_turn_fact",
            "atdtmco_ns_fact",
        ]
    ].copy()
    numeric_columns = [
        "atdtmco_saldo_turn_pred",
        "atdtmco_ns_pred",
        "atdtmco_saldo_turn_fact",
        "atdtmco_ns_fact",
    ]
    if oracle_export_df.duplicated(
        ["score_date", "atdtmco_cashdesk_name", "forecast_date"]
    ).any():
        raise ValueError("Перед записью найдены дубли прогнозов")
    if not np.isfinite(oracle_export_df[numeric_columns].to_numpy(dtype=float)).all():
        raise ValueError("Перед записью найдены невалидные числовые значения")

    oracle_export_df[numeric_columns] = oracle_export_df[numeric_columns].round(2)
    if engine_cdw is None:
        engine_cdw = await create_cdw_engine()
    oracle.write(
        oracle_export_df,
        engine_cdw,
        ORACLE_TARGET_TABLE,
        batch_size=100_000,
        if_exists="append",
    )
    print(f"В {ORACLE_TARGET_TABLE} записано {len(oracle_export_df):,} строк")
else:
    print("Запись в Oracle отключена: WRITE_TO_ORACLE = False")